In [1]:
from aviary.core import Environment, Message, Tool, ToolRequestMessage,ToolResponseMessage
from ldp.agent import Agent
from ldp.graph import LLMCallOp
import os
from helper.problem import examples
from helper.units import UNIT_ALIASES
from helper.tools import *
from typing import Any, Tuple, List
from pydantic import BaseModel, Field
from ldp.graph import LLMCallOp, OpResult, compute_graph

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [3]:
# --- Step 1: Science (more like chemistry) Environment -> ---
class ChemistryEnv(Environment[ProblemState]):
    def __init__(self, problem: ProblemState):
          super().__init__()
          self.problem = problem
          self._oracle = None
        
    async def reset(self):
        self.tools = [Tool.from_function(fun) 
                      for _, fun in TOOLS.items()]
        print(f"[self.problem]: {self.problem}")
        prompt = self.problem.prompt or build_prompt(self.problem)
        print(f"[RESET] prompt: {prompt}")
        return [Message(content=prompt)], self.tools

    async def step(self, action):
        # 1) If it’s a tool request, run it and return *native* ToolResponseMessage(s)
        if isinstance(action, ToolRequestMessage) and action.tool_calls:
            raw = await self.exec_tool_calls(action)
            # raw may be one ToolResponseMessage or a list of them
            if isinstance(raw, ToolResponseMessage):
                obs = [raw]
            elif isinstance(raw, list) and all(isinstance(r, ToolResponseMessage) for r in raw):
                obs = raw
            else:
                # (shouldn’t happen—just in case)
                obs = [Message(content=str(raw))]
            print(f"[DEBUG]: {obs}")
            return obs, 0.0, False, False
        
        # 2) Otherwise `action` is the final assistant Message—score and finish
        assert isinstance(action, Message) and action.content is not None
        reward = float(action.content.strip() == self.problem.expected.strip())
        return [action], reward, True, False


In [4]:
# --- Step 2: Guided Agent ---
# can decide what tools to use at each step without using tool selector
# uses compute graph

class GuidedAgentState(BaseModel):
    messages: List[Message]
    tools:    List[Tool]

    def get_next_state(self, obs: List[Message]) -> "GuidedAgentState":
        new_msgs = list(self.messages)

        for item in obs:
            if isinstance(item, Message):
                new_msgs.append(item)

            elif hasattr(item, "content"):
                new_msgs.append(Message(content=item.content))

            elif isinstance(item, dict):
                new_msgs.append(Message(**item))

            else:
                continue
        print(f"[DEBUG]: {new_msgs}")
        return GuidedAgentState(messages=new_msgs, tools=self.tools)

class GuidedAgent(BaseModel, Agent[GuidedAgentState]):
    llm_model: dict[str, Any] = Field(
        default={
            "name": "gpt-4o-2024-08-06",
            "temperature": 0.1,
        }
    )
    guidelines_msg: Message = Field(
        default=Message(
            role="system",
            content=(
                "You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\n"
                "Step by step:\n"
                "  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n"
                "  2) Observe the tool’s output (e.g. “0.175 M”).\n"
                "  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n"
                "  4) Observe and return that final tool output as your answer."
            )
        ),
        description="System‐level instructions for the LLM.",
    )

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._llm_call_op = LLMCallOp()

    async def init_state(self, tools: List[Tool]) -> GuidedAgentState:
        return GuidedAgentState(messages=[self.guidelines_msg], tools=tools)

    @compute_graph()
    async def get_asv(self, agent_state: GuidedAgentState, obs: List[Message]) -> Tuple[
          OpResult[ToolRequestMessage], 
          GuidedAgentState,
          float                          
        ]:
        # 1) incorporate the new environment observation
        next_state = agent_state.get_next_state(obs)

        # 2) call the LLM to pick exactly one tool + args
        llm_result: OpResult[ToolRequestMessage] = await self._llm_call_op(
            self.llm_model,
            [self.guidelines_msg, *next_state.messages],
            next_state.tools
        )

        # 3) append the tool call into the agent’s message history
        next_state.messages.append(llm_result.value)

        # 4) return the op result
        return llm_result, next_state, 0.0

In [5]:
agent = GuidedAgent()
MAX_CALLS = 5

for problem in examples:
    env = ChemistryEnv(problem)
    obs, tools = await env.reset()
    state = await agent.init_state(tools)

    print("\n--- Chain‐of‐Thought ---")
    for step in range(1, MAX_CALLS + 1):
        # 1) Ask the agent for its next move
        op_result, state, _ = await agent.get_asv(state, obs)
        action = op_result.value
        print(op_result)
        print(op_result.value)

        # 2) Execute in the env
        obs, _, _, _ = await env.step(action)
        tool_output = obs[0].content.strip()
        print("[Tool output]", tool_output)
        print(f"[Expected]: {problem.expected.strip()}")

        # 3) If the answer is here, stop
        if tool_output == problem.expected.strip():
            print("\n[Final answer]", tool_output)
            print("Expected:", problem.expected)
            print("Reward:", float(tool_output == problem.expected.strip()))
            break


[self.problem]: prompt=None task='molar_mass' params={'formula': 'H2O'} expected='18.02 g/mol'
[DEBUG build_prompt] task='molar_mass', params={'formula': 'H2O'}
[RESET] prompt: What is the molar mass of H2O?

--- Chain‐of‐Thought ---
[DEBUG]: [Message(role='system', content='You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\nStep by step:\n  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n  2) Observe the tool’s output (e.g. “0.175 M”).\n  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n  4) Observe and return that final tool output as your answer.'), Message(role='user', content='What is the molar mass of H2O?')]
Tool request message '' for tool calls: lookup_molar_mass(formula='H2O') [id=call_UmiqAbiu7yLWdXFzbYVPYrVa]
Tool request message '' for tool calls: lookup_molar_mass(formula='H2O') [id=call_UmiqAbiu7yLWdXFzbYVPYrVa]
[DEBUG]: [ToolResponseMessage(role='too

15:45:26 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:26 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:26 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:26 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


Tool request message '' for tool calls: calc(Expression='2.26 + 0.5') [id=call_Y6EJRx6Sb1bQT3pEjOtEcKpH]
Tool request message '' for tool calls: calc(Expression='2.26 + 0.5') [id=call_Y6EJRx6Sb1bQT3pEjOtEcKpH]
[DEBUG]: [ToolResponseMessage(role='tool', content='2.76', name='calc', tool_call_id='call_Y6EJRx6Sb1bQT3pEjOtEcKpH')]
[Tool output] 2.76
[Expected]: 2.27 L
[DEBUG]: [Message(role='system', content='You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\nStep by step:\n  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n  2) Observe the tool’s output (e.g. “0.175 M”).\n  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n  4) Observe and return that final tool output as your answer.'), Message(role='user', content='You dissolve 12.0 g of Na₂CO₃ (M = 105.99 g/mol) to make 0.500 L of solution. To obtain a 0.050 M solution, to what **total volume** should you dilute the s

15:45:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:27 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


Tool request message '' for tool calls: dilution(initial_conc='0.2264364562694594', initial_vol='0.5', final_conc='0.05') [id=call_UtKsyIM8CK3rJuuPXhptdB3v]
Tool request message '' for tool calls: dilution(initial_conc='0.2264364562694594', initial_vol='0.5', final_conc='0.05') [id=call_UtKsyIM8CK3rJuuPXhptdB3v]
[DEBUG]: [ToolResponseMessage(role='tool', content='2.26 L', name='dilution', tool_call_id='call_UtKsyIM8CK3rJuuPXhptdB3v')]
[Tool output] 2.26 L
[Expected]: 2.27 L
[self.problem]: prompt='A cylinder contains 500 cm³ of a gas at STP. The gas density at STP is 1.96 g/L. What is the **mass** of gas in the cylinder? Give only the final answer with units (g).' task=None params={'volume_cm3': 500.0, 'density_g_per_L': 1.96, 'stp_molar_volume_L_per_mol': 22.4} expected='0.98 g'
[RESET] prompt: A cylinder contains 500 cm³ of a gas at STP. The gas density at STP is 1.96 g/L. What is the **mass** of gas in the cylinder? Give only the final answer with units (g).

--- Chain‐of‐Thought --

...

Tool request message '' for tool calls: dilution(initial_conc='0.1', initial_vol='0.035', final_conc='0.13') [id=call_wpHvOXrl3M0CQHLkf5ZgzOts]
Tool request message '' for tool calls: dilution(initial_conc='0.1', initial_vol='0.035', final_conc='0.13') [id=call_wpHvOXrl3M0CQHLkf5ZgzOts]
[DEBUG]: [ToolResponseMessage(role='tool', content='0.03 L', name='dilution', tool_call_id='call_wpHvOXrl3M0CQHLkf5ZgzOts')]
[Tool output] 0.03 L
[Expected]: 2.00
[DEBUG]: [Message(role='system', content='You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\nStep by step:\n  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n  2) Observe the tool’s output (e.g. “0.175 M”).\n  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n  4) Observe and return that final tool output as your answer.'), Message(role='user', content='You titrate 20.0 mL of 0.100 M HCl with 0.100 M NaOH. After adding 15.0 

15:45:46 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:46 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:46 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:46 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


Tool request message '' for tool calls: pH_from_H(H_conc='0.03') [id=call_vpN8kvEnbA0V8ptt1qeenUMF]
Tool request message '' for tool calls: pH_from_H(H_conc='0.03') [id=call_vpN8kvEnbA0V8ptt1qeenUMF]
[DEBUG]: [ToolResponseMessage(role='tool', content='1.52 pH', name='pH_from_H', tool_call_id='call_vpN8kvEnbA0V8ptt1qeenUMF')]
[Tool output] 1.52 pH
[Expected]: 2.00
[DEBUG]: [Message(role='system', content='You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\nStep by step:\n  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n  2) Observe the tool’s output (e.g. “0.175 M”).\n  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n  4) Observe and return that final tool output as your answer.'), Message(role='user', content='You titrate 20.0 mL of 0.100 M HCl with 0.100 M NaOH. After adding 15.0 mL of NaOH, you dilute the 35.0 mL mixture to a total volume of 50.0 mL. What is the 

15:45:49 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:49 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:49 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
15:45:49 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30


Tool request message '' for tool calls: calc(expression='1.52') [id=call_048NElTW7OhUYMXm1FMs2UtD]
Tool request message '' for tool calls: calc(expression='1.52') [id=call_048NElTW7OhUYMXm1FMs2UtD]
[DEBUG]: [ToolResponseMessage(role='tool', content='1.52', name='calc', tool_call_id='call_048NElTW7OhUYMXm1FMs2UtD')]
[Tool output] 1.52
[Expected]: 2.00
[DEBUG]: [Message(role='system', content='You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\nStep by step:\n  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n  2) Observe the tool’s output (e.g. “0.175 M”).\n  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n  4) Observe and return that final tool output as your answer.'), Message(role='user', content='You titrate 20.0 mL of 0.100 M HCl with 0.100 M NaOH. After adding 15.0 mL of NaOH, you dilute the 35.0 mL mixture to a total volume of 50.0 mL. What is the **final pH** 